In [ ]:
import os
import cv2
import numpy as np

class PlateCellLabeler:
    def __init__(self, image_path, rows, cols, output_dir="dataset"):
        self.image_path = image_path
        self.rows = rows
        self.cols = cols
        self.output_dir = output_dir
        
        # Create output directories
        os.makedirs(os.path.join(output_dir, "alive"), exist_ok=True)
        os.makedirs(os.path.join(output_dir, "dead"), exist_ok=True)
        
        # Load image
        self.original_img = cv2.imread(image_path)
        if self.original_img is None:
            raise ValueError(f"Could not load image from {image_path}")
            
        self.img_height, self.img_width = self.original_img.shape[:2]
        self.cell_height = self.img_height / rows
        self.cell_width = self.img_width / cols
        
        # State tracking
        self.selected_r = None
        self.selected_c = None
        self.counts = {"alive": 0, "dead": 0}
        
        # Counters for labeled status to avoid overwriting
        self.base_name = os.path.splitext(os.path.basename(image_path))[0]

    def mouse_callback(self, event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN:
            # Determine which row and column was clicked
            c = int(x // self.cell_width)
            r = int(y // self.cell_height)
            
            if 0 <= r < self.rows and 0 <= c < self.cols:
                self.selected_r = r
                self.selected_c = c
                print(f"Selected Cell -> Row: {r}, Col: {c}")

    def run(self):
        cv2.namedWindow("Plate Labeler", cv2.WINDOW_NORMAL)
        cv2.setMouseCallback("Plate Labeler", self.mouse_callback)
        
        print("\n--- Controls ---")
        print("1. Click a cell to select it.")
        print("2. Press 'a' to save selection as ALIVE.")
        print("3. Press 'd' to save selection as DEAD.")
        print("4. Press 'q' to quit.")
        print("----------------\n")
        
        while True:
            # Create a fresh copy of the image for drawing the UI overlay
            display_img = self.original_img.copy()
            
            # Draw the grid lines
            for r in range(1, self.rows):
                pt1 = (0, int(r * self.cell_height))
                pt2 = (self.img_width, int(r * self.cell_height))
                cv2.line(display_img, pt1, pt2, (200, 200, 200), 1)
                
            for c in range(1, self.cols):
                pt1 = (int(c * self.cell_width), 0)
                pt2 = (int(c * self.cell_width), self.img_height)
                cv2.line(display_img, pt1, pt2, (200, 200, 200), 1)
                
            # Highlight currently selected cell (if any) in Blue
            if self.selected_r is not None and self.selected_c is not None:
                start_x = int(self.selected_c * self.cell_width)
                start_y = int(self.selected_r * self.cell_height)
                end_x = int((self.selected_c + 1) * self.cell_width)
                end_y = int((self.selected_r + 1) * self.cell_height)
                
                # Draw blue bounding box for selection
                cv2.rectangle(display_img, (start_x, start_y), (end_x, end_y), (255, 0, 0), 2)
            
            # Show status text on image
            status_text = f"Alive: {self.counts['alive']} | Dead: {self.counts['dead']}"
            cv2.putText(display_img, status_text, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            
            cv2.imshow("Plate Labeler", display_img)
            key = cv2.waitKey(1) & 0xFF
            
            # Quit
            if key == ord('q'):
                break
                
            # Save as Alive
            elif key == ord('a') and self.selected_r is not None:
                self._save_crop("alive")
                
            # Save as Dead
            elif key == ord('d') and self.selected_r is not None:
                self._save_crop("dead")
                
        cv2.destroyAllWindows()

    def _save_crop(self, label):
        r, c = self.selected_r, self.selected_c
        
        # Calculate precise pixel coordinates for cropping
        start_y = int(r * self.cell_height)
        end_y = int((r + 1) * self.cell_height) if r < self.rows - 1 else self.img_height
        
        start_x = int(c * self.cell_width)
        end_x = int((c + 1) * self.cell_width) if c < self.cols - 1 else self.img_width
        
        # Crop from original image
        cell_crop = self.original_img[start_y:end_y, start_x:end_x]
        
        if cell_crop.size > 0:
            filename = f"{self.base_name}_r{r}_c{c}.png"
            output_path = os.path.join(self.output_dir, label, filename)
            cv2.imwrite(output_path, cell_crop)
            self.counts[label] += 1
            print(f"-> Saved [ {label.upper()} ]: {output_path}")
        else:
            print("-> Error: Invalid crop region.")
            
        # Reset selection
        self.selected_r = None
        self.selected_c = None

# ==========================================
# How to run:
# ==========================================
if __name__ == "__main__":
    # Change these to match your plate image and grid dimensions
    IMAGE_PATH = "plate_mixed_sample.jpg"
    TOTAL_ROWS = 12
    TOTAL_COLS = 8
    
    labeler = PlateCellLabeler(IMAGE_PATH, rows=TOTAL_ROWS, cols=TOTAL_COLS)
    labeler.run()